In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
carpeta = Path('Datos_Historicos')

dfs = {
    archivo.stem: pd.read_csv(
        archivo, sep=';', decimal=',',
        dtype={'FECHA (YYMMDD)': str, 'HORA (HHMM)': str},
    ).dropna(axis=1, how='all')
    for archivo in sorted(carpeta.glob('*.csv'))
}

for nombre, df in dfs.items():
    df['fecha_hora'] = pd.to_datetime(
        df['FECHA (YYMMDD)'] + df['HORA (HHMM)'], format='%y%m%d%H%M'
    )
    dfs[nombre] = df.drop(columns=['FECHA (YYMMDD)', 'HORA (HHMM)']).set_index('fecha_hora')


In [3]:
def graficar(nombre, desde=None, hasta=None, frecuencia='D'):
    df = dfs[nombre].loc[desde:hasta]
    if frecuencia:
        df = df.resample(frecuencia).mean()

    fig, ax = plt.subplots(figsize=(14, 5))
    for columna in df.columns:
        ax.plot(df.index, df[columna], linewidth=0.8, label=columna)

    ax.set_title(f"MP2.5 - {nombre.replace('_', ' ')}")
    ax.set_xlabel('Fecha')
    ax.set_ylabel('MP2.5 (µg/m³)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
